# Feature Engineering Worksheet
## Transforming Raw Data into Powerful Features

Feature engineering is often considered the key to successful machine learning. In this workshop, you'll learn:

1. **Feature Scaling & Normalization**
2. **Polynomial & Interaction Features** 
3. **Categorical Encoding Techniques**
4. **Feature Selection Methods**
5. **Creating Features from DateTime & Text**
6. **Real-world Examples & Best Practices**

**"Raw data is like a diamond in the rough - feature engineering is the art of cutting and polishing it."**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, fetch_california_housing, load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, 
    PolynomialFeatures, LabelEncoder, OneHotEncoder
)
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, SelectFromModel
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
np.random.seed(42)

## 1. Feature Scaling and Normalization

Different algorithms are sensitive to feature scales. Let's explore various scaling techniques.

In [ ]:
# Create sample data with different scales
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.normal(35, 10, n_samples),  # 20-50 range
    'salary': np.random.normal(50000, 15000, n_samples),  # 20k-80k range
    'experience': np.random.normal(8, 4, n_samples),  # 0-16 range
    'rating': np.random.uniform(1, 5, n_samples)  # 1-5 range
}

# Add some correlation
data['salary'] += data['experience'] * 2000
data['rating'] += data['experience'] * 0.1

df_original = pd.DataFrame(data)

# Display basic statistics
print("Original Data Statistics:")
print(df_original.describe())

# Visualize the original distributions
plt.figure(figsize=(16, 4))

for i, col in enumerate(df_original.columns):
    plt.subplot(1, 4, i+1)
    plt.hist(df_original[col], bins=30, alpha=0.7, color=f'C{i}')
    plt.title(f'{col}\n(μ={df_original[col].mean():.1f}, σ={df_original[col].std():.1f})')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

### Different Scaling Techniques

In [ ]:
# Apply different scaling techniques
scalers = {
    'StandardScaler': StandardScaler(),  # (X - mean) / std
    'MinMaxScaler': MinMaxScaler(),      # (X - min) / (max - min)
    'RobustScaler': RobustScaler()       # (X - median) / IQR
}

scaled_data = {}
for name, scaler in scalers.items():
    scaled_data[name] = pd.DataFrame(
        scaler.fit_transform(df_original), 
        columns=df_original.columns
    )

# Visualize scaling effects
fig, axes = plt.subplots(4, 4, figsize=(16, 12))

datasets = {'Original': df_original, **scaled_data}

for row, (dataset_name, df) in enumerate(datasets.items()):
    for col, column in enumerate(df.columns):
        ax = axes[row, col]
        ax.hist(df[column], bins=30, alpha=0.7, color=f'C{col}')
        ax.set_title(f'{dataset_name}: {column}')
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Compare statistics
print("\nScaling Comparison (means and stds):")
for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"Means: {df.mean().round(3).values}")
    print(f"Stds:  {df.std().round(3).values}")

### Impact of Scaling on Model Performance

In [ ]:
# Create a simple regression problem
X, y = make_regression(n_samples=1000, n_features=4, noise=0.1, random_state=42)

# Make features have different scales
X[:, 0] *= 100      # Large scale
X[:, 1] *= 0.01     # Small scale
X[:, 2] *= 10       # Medium scale
# X[:, 3] stays as is

feature_names = ['Large_Scale', 'Small_Scale', 'Medium_Scale', 'Normal_Scale']
X_df = pd.DataFrame(X, columns=feature_names)

print("Feature Scales:")
print(X_df.describe())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Test different models with and without scaling
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

models = {
    'Linear Regression': LinearRegression(),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(kernel='rbf')
}

results = []

for model_name, model in models.items():
    # Without scaling
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2_no_scaling = r2_score(y_test, y_pred)
    
    # With standard scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model.fit(X_train_scaled, y_train)
    y_pred_scaled = model.predict(X_test_scaled)
    r2_with_scaling = r2_score(y_test, y_pred_scaled)
    
    results.append({
        'Model': model_name,
        'R² (No Scaling)': r2_no_scaling,
        'R² (With Scaling)': r2_with_scaling,
        'Improvement': r2_with_scaling - r2_no_scaling
    })

results_df = pd.DataFrame(results)
print("\nScaling Impact on Model Performance:")
print(results_df.round(4))

## 2. Polynomial and Interaction Features

Creating non-linear features from linear ones can dramatically improve model performance.

In [ ]:
# Create a non-linear relationship
np.random.seed(42)
X_simple = np.random.uniform(-2, 2, (300, 2))
y_nonlinear = (
    X_simple[:, 0]**2 + 
    X_simple[:, 1]**2 + 
    X_simple[:, 0] * X_simple[:, 1] + 
    np.random.normal(0, 0.1, 300)
)

# Create DataFrame
df_poly = pd.DataFrame(X_simple, columns=['x1', 'x2'])
df_poly['y'] = y_nonlinear

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y_nonlinear, test_size=0.3, random_state=42
)

# Visualize the relationship
fig = plt.figure(figsize=(15, 5))

# 3D plot
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(df_poly['x1'], df_poly['x2'], df_poly['y'], c=df_poly['y'], cmap='viridis')
ax1.set_xlabel('x1')
ax1.set_ylabel('x2')
ax1.set_zlabel('y')
ax1.set_title('Original Non-linear Relationship')

# 2D projections
plt.subplot(132)
plt.scatter(df_poly['x1'], df_poly['y'], alpha=0.6, c='blue')
plt.xlabel('x1')
plt.ylabel('y')
plt.title('y vs x1')

plt.subplot(133)
plt.scatter(df_poly['x2'], df_poly['y'], alpha=0.6, c='red')
plt.xlabel('x2')
plt.ylabel('y')
plt.title('y vs x2')

plt.tight_layout()
plt.show()

### Comparing Linear vs Polynomial Features

In [ ]:
# Compare linear vs polynomial regression
degrees = [1, 2, 3, 4]
poly_results = []

plt.figure(figsize=(16, 10))

for i, degree in enumerate(degrees):
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train_poly)
    y_pred_test = model.predict(X_test_poly)
    
    # Calculate scores
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    poly_results.append({
        'Degree': degree,
        'Features': X_train_poly.shape[1],
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Overfitting': train_r2 - test_r2
    })
    
    # Create visualization mesh for degree 2
    if degree == 2:
        # Create mesh for prediction surface
        x1_range = np.linspace(X_simple[:, 0].min(), X_simple[:, 0].max(), 50)
        x2_range = np.linspace(X_simple[:, 1].min(), X_simple[:, 1].max(), 50)
        X1_mesh, X2_mesh = np.meshgrid(x1_range, x2_range)
        X_mesh = np.c_[X1_mesh.ravel(), X2_mesh.ravel()]
        X_mesh_poly = poly.transform(X_mesh)
        y_mesh_pred = model.predict(X_mesh_poly).reshape(X1_mesh.shape)
    
    # Plot results
    plt.subplot(2, 4, i+1)
    plt.scatter(y_test, y_pred_test, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('True Values')
    plt.ylabel('Predictions')
    plt.title(f'Degree {degree}\nTest R² = {test_r2:.3f}')
    
    # Show feature names for degree 2
    if degree == 2:
        feature_names = poly.get_feature_names_out(['x1', 'x2'])
        plt.subplot(2, 4, i+5)
        coef_importance = np.abs(model.coef_)
        plt.bar(range(len(coef_importance)), coef_importance)
        plt.xticks(range(len(feature_names)), feature_names, rotation=45)
        plt.ylabel('Coefficient Magnitude')
        plt.title('Feature Importance (Degree 2)')
        
        print(f"\nDegree {degree} Features: {feature_names}")
        print(f"Coefficients: {model.coef_.round(3)}")

# Plot prediction surface for degree 2
if 'y_mesh_pred' in locals():
    plt.subplot(2, 4, 7)
    contour = plt.contourf(X1_mesh, X2_mesh, y_mesh_pred, levels=20, alpha=0.8, cmap='viridis')
    plt.scatter(X_simple[:, 0], X_simple[:, 1], c=y_nonlinear, cmap='viridis', edgecolors='black')
    plt.colorbar(contour)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Degree 2 Prediction Surface')

# Performance comparison
plt.subplot(2, 4, 8)
degrees_list = [r['Degree'] for r in poly_results]
train_r2s = [r['Train R²'] for r in poly_results]
test_r2s = [r['Test R²'] for r in poly_results]

plt.plot(degrees_list, train_r2s, 'o-', label='Train R²', linewidth=2)
plt.plot(degrees_list, test_r2s, 's-', label='Test R²', linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('Train vs Test Performance')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Results table
poly_df = pd.DataFrame(poly_results)
print("\nPolynomial Degree Comparison:")
print(poly_df.round(4))

## 3. Categorical Encoding Techniques

Machine learning models work with numbers, so we need to convert categorical variables appropriately.

In [ ]:
  # Create sample categorical data
  np.random.seed(42)
  n_samples = 1000

  categorical_data = {
      'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix'], n_samples),
      'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples,
                                  p=[0.4, 0.3, 0.2, 0.1]),
      'color': np.random.choice(['Red', 'Green', 'Blue'], n_samples),
      'size': np.random.choice(['Small', 'Medium', 'Large'], n_samples, p=[0.3, 0.5, 0.2])
  }

  # Create target variable with some relationship to categories
  city_effect = {'NYC': 10, 'LA': 8, 'Chicago': 6, 'Houston': 5, 'Phoenix': 4}
  education_effect = {'High School': 0, 'Bachelor': 5, 'Master': 8, 'PhD': 12}
  size_effect = {'Small': -2, 'Medium': 0, 'Large': 3}

  y_categorical = (
      np.array([city_effect[city] for city in categorical_data['city']]) +
      np.array([education_effect[edu] for edu in categorical_data['education']]) +
      np.array([size_effect[size] for size in categorical_data['size']]) +
      np.random.normal(0, 2, n_samples)
  )

  df_categorical = pd.DataFrame(categorical_data)
  df_categorical['target'] = y_categorical

  print("Categorical Data Sample:")
  print(df_categorical.head(10))

  # Visualize categorical distributions
  plt.figure(figsize=(16, 8))

  categorical_cols = ['city', 'education', 'color', 'size']
  for i, col in enumerate(categorical_cols):
      plt.subplot(2, 4, i+1)
      df_categorical[col].value_counts().plot(kind='bar')
      plt.title(f'{col.title()} Distribution')
      plt.xticks(rotation=45)

      # Box plot showing relationship with target
      plt.subplot(2, 4, i+5)
      df_categorical.boxplot(column='target', by=col, ax=plt.gca())
      plt.title(f'Target vs {col.title()}')
      plt.xticks(rotation=45)

  plt.tight_layout()
  plt.show()

### Different Encoding Techniques

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder, BinaryEncoder

# Prepare data
X_cat = df_categorical[categorical_cols]
y_cat = df_categorical['target']

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y_cat, test_size=0.3, random_state=42
)

# Different encoding techniques
encoders = {
    'Label Encoding': LabelEncoder(),
    'One-Hot Encoding': OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
    'Ordinal Encoding': OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
    'Target Encoding': TargetEncoder(),
}

encoding_results = []

print("Encoding Comparison:")
print("=" * 50)

for encoding_name, encoder in encoders.items():
    print(f"\n{encoding_name}:")
    
    if encoding_name == 'Label Encoding':
        # Label encoding for each column separately
        X_train_encoded = X_train_cat.copy()
        X_test_encoded = X_test_cat.copy()
        
        for col in categorical_cols:
            le = LabelEncoder()
            X_train_encoded[col] = le.fit_transform(X_train_cat[col])
            # Handle unknown categories in test set
            X_test_encoded[col] = X_test_cat[col].map(
                dict(zip(le.classes_, le.transform(le.classes_)))
            ).fillna(-1).astype(int)
            
            print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    
    elif encoding_name == 'Target Encoding':
        X_train_encoded = encoder.fit_transform(X_train_cat, y_train_cat)
        X_test_encoded = encoder.transform(X_test_cat)
        print(f"  Features created: {X_train_encoded.shape[1]}")
    
    else:
        X_train_encoded = encoder.fit_transform(X_train_cat)
        X_test_encoded = encoder.transform(X_test_cat)
        
        if encoding_name == 'One-Hot Encoding':
            feature_names = encoder.get_feature_names_out(categorical_cols)
            print(f"  Features created: {len(feature_names)}")
            print(f"  Sample features: {feature_names[:10]}...")
        else:
            print(f"  Features created: {X_train_encoded.shape[1]}")
    
    # Train a simple model to compare performance
    model = LinearRegression()
    model.fit(X_train_encoded, y_train_cat)
    
    y_pred = model.predict(X_test_encoded)
    r2 = r2_score(y_test_cat, y_pred)
    mse = mean_squared_error(y_test_cat, y_pred)
    
    encoding_results.append({
        'Encoding': encoding_name,
        'Features': X_train_encoded.shape[1],
        'R²': r2,
        'MSE': mse
    })
    
    print(f"  R² Score: {r2:.4f}")
    print(f"  MSE: {mse:.4f}")

# Results comparison
encoding_df = pd.DataFrame(encoding_results)
print("\n\nEncoding Performance Comparison:")
print(encoding_df.round(4))

### Visualizing Encoding Effects

In [ ]:
# Visualize different encoding effects for education column
plt.figure(figsize=(16, 10))

# Original relationship
plt.subplot(2, 3, 1)
education_means = df_categorical.groupby('education')['target'].mean().sort_values()
education_means.plot(kind='bar')
plt.title('Original: Education vs Target')
plt.xticks(rotation=45)
plt.ylabel('Average Target')

# Label Encoding
le_education = LabelEncoder()

education_labels = le_education.fit_transform(df_categorical['education'])
plt.subplot(2, 3, 2)
plt.scatter(education_labels, df_categorical['target'], alpha=0.5)
plt.xlabel('Label Encoded Education')
plt.ylabel('Target')
plt.title('Label Encoding')
education_mapping = dict(zip(le_education.classes_, le_education.transform(le_education.classes_)))
plt.xticks(list(education_mapping.values()), list(education_mapping.keys()), rotation=45)

# One-Hot Encoding visualization
ohe_education = OneHotEncoder(sparse_output=False)
education_onehot = ohe_education.fit_transform(df_categorical[['education']])
education_features = ohe_education.get_feature_names_out(['education'])

plt.subplot(2, 3, 3)
correlations = []
for i, feature in enumerate(education_features):
    corr = np.corrcoef(education_onehot[:, i], df_categorical['target'])[0, 1]
    correlations.append(corr)

plt.bar(range(len(education_features)), correlations)
plt.xticks(range(len(education_features)), 
           [f.replace('education_', '') for f in education_features], rotation=45)
plt.ylabel('Correlation with Target')
plt.title('One-Hot: Feature Correlations')

# Target Encoding
target_encoder = TargetEncoder()
education_target_encoded = target_encoder.fit_transform(
    df_categorical[['education']], df_categorical['target']
)
plt.subplot(2, 3, 4)
plt.scatter(education_target_encoded['education'], df_categorical['target'], alpha=0.5)
plt.xlabel('Target Encoded Education')
plt.ylabel('Target')
plt.title('Target Encoding')

# Feature count comparison
plt.subplot(2, 3, 5)
feature_counts = [1, 1, len(education_features), 1]  # Original, Label, One-Hot, Target
encoding_names = ['Original', 'Label', 'One-Hot', 'Target']
plt.bar(encoding_names, feature_counts)
plt.ylabel('Number of Features')
plt.title('Feature Count by Encoding')

# Performance comparison
plt.subplot(2, 3, 6)
r2_scores = [result['R²'] for result in encoding_results]
encoding_labels = [result['Encoding'] for result in encoding_results]
plt.bar(range(len(r2_scores)), r2_scores)
plt.xticks(range(len(encoding_labels)), encoding_labels, rotation=45)
plt.ylabel('R² Score')
plt.title('Model Performance by Encoding')

plt.tight_layout()
plt.show()

## 4. Feature Selection Techniques

Not all features are useful. Let's learn how to select the most important ones.

In [ ]:
# Create a dataset with many features, some relevant, some not
from sklearn.datasets import make_regression

X_many, y_many = make_regression(
    n_samples=1000, 
    n_features=50,
    n_informative=10,  # Only 10 features are actually informative
    noise=0.1,
    random_state=42
)

feature_names = [f'feature_{i}' for i in range(X_many.shape[1])]
X_many_df = pd.DataFrame(X_many, columns=feature_names)

X_train_many, X_test_many, y_train_many, y_test_many = train_test_split(
    X_many, y_many, test_size=0.3, random_state=42
)

print(f"Dataset shape: {X_many.shape}")
print(f"Informative features: 10")
print(f"Redundant features: 5")
print(f"Noise features: {50 - 10 - 5} = 35")

# Baseline performance with all features
baseline_model = LinearRegression()
baseline_model.fit(X_train_many, y_train_many)
baseline_pred = baseline_model.predict(X_test_many)
baseline_r2 = r2_score(y_test_many, baseline_pred)

print(f"\nBaseline R² (all 50 features): {baseline_r2:.4f}")

### Univariate Feature Selection

In [ ]:
# Univariate feature selection methods
selection_methods = {
    'f_regression': SelectKBest(score_func=f_regression, k=15),
    'mutual_info': SelectKBest(score_func=mutual_info_regression, k=15)
}

univariate_results = []

plt.figure(figsize=(16, 8))

for i, (method_name, selector) in enumerate(selection_methods.items()):
    # Fit selector
    X_train_selected = selector.fit_transform(X_train_many, y_train_many)
    X_test_selected = selector.transform(X_test_many)
    
    # Get scores and selected features
    scores = selector.scores_
    selected_features = selector.get_support(indices=True)
    
    # Train model on selected features
    model = LinearRegression()
    model.fit(X_train_selected, y_train_many)
    y_pred_selected = model.predict(X_test_selected)
    r2_selected = r2_score(y_test_many, y_pred_selected)
    
    univariate_results.append({
        'Method': method_name,
        'Features Selected': len(selected_features),
        'R²': r2_selected,
        'Selected Features': selected_features.tolist()
    })
    
    # Plot feature scores
    plt.subplot(2, 3, i+1)
    plt.bar(range(len(scores)), scores)
    plt.axhline(y=np.sort(scores)[-15], color='red', linestyle='--', 
                label=f'Top 15 threshold')
    plt.xlabel('Feature Index')
    plt.ylabel('Score')
    plt.title(f'{method_name.replace("_", " ").title()} Scores')
    plt.legend()
    
    # Plot selected vs non-selected
    plt.subplot(2, 3, i+3)
    selected_mask = np.zeros(len(scores), dtype=bool)
    selected_mask[selected_features] = True
    
    plt.scatter(range(len(scores)), scores, 
               c=['red' if selected else 'blue' for selected in selected_mask],
               alpha=0.6)
    plt.xlabel('Feature Index')
    plt.ylabel('Score')
    plt.title(f'{method_name}: Selected (Red) vs Not Selected (Blue)')
    
    print(f"\n{method_name.replace('_', ' ').title()}:")
    print(f"  R² Score: {r2_selected:.4f}")
    print(f"  Top 10 features: {sorted(selected_features)[:10]}")

plt.tight_layout()
plt.show()

### Model-Based Feature Selection

In [ ]:
# Model-based feature selection
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor

model_based_methods = {
    'Lasso': SelectFromModel(Lasso(alpha=0.1)),
    'Random Forest': SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42)),
    'RFE (Linear)': RFE(LinearRegression(), n_features_to_select=15)
}

model_based_results = []

plt.figure(figsize=(18, 6))

for i, (method_name, selector) in enumerate(model_based_methods.items()):
    # Fit selector
    X_train_selected = selector.fit_transform(X_train_many, y_train_many)
    X_test_selected = selector.transform(X_test_many)
    
    # Get selected features
    selected_features = selector.get_support(indices=True)
    
    # Train model on selected features
    model = LinearRegression()
    model.fit(X_train_selected, y_train_many)
    y_pred_selected = model.predict(X_test_selected)
    r2_selected = r2_score(y_test_many, y_pred_selected)
    
    model_based_results.append({
        'Method': method_name,
        'Features Selected': len(selected_features),
        'R²': r2_selected,
        'Selected Features': selected_features.tolist()
    })
    
    # Get feature importances
    if method_name == 'Random Forest':
        importances = selector.estimator_.feature_importances_
    elif method_name == 'Lasso':
        importances = np.abs(selector.estimator_.coef_)
    elif method_name == 'RFE (Linear)':
        importances = selector.ranking_  # Lower ranking = more important
        importances = 1.0 / importances  # Invert for plotting
    
    # Plot feature importances
    plt.subplot(1, 3, i+1)
    colors = ['red' if idx in selected_features else 'blue' 
              for idx in range(len(importances))]
    plt.bar(range(len(importances)), importances, color=colors, alpha=0.7)
    plt.xlabel('Feature Index')
    plt.ylabel('Importance')
    plt.title(f'{method_name}\nR² = {r2_selected:.4f}, Features = {len(selected_features)}')
    
    print(f"\n{method_name}:")
    print(f"  R² Score: {r2_selected:.4f}")
    print(f"  Features selected: {len(selected_features)}")
    print(f"  Top 10 features: {sorted(selected_features)[:10]}")

plt.tight_layout()
plt.show()

### Feature Selection Performance Comparison

In [ ]:
# Combine all results
all_selection_results = (
    [{'Method': 'Baseline (All)', 'Features Selected': 50, 'R²': baseline_r2}] +
    univariate_results + 
    model_based_results
)

selection_df = pd.DataFrame(all_selection_results)
print("Feature Selection Results:")
print(selection_df[['Method', 'Features Selected', 'R²']].round(4))

# Visualize results
plt.figure(figsize=(14, 6))

# Performance vs number of features
plt.subplot(1, 2, 1)
plt.scatter(selection_df['Features Selected'], selection_df['R²'], 
           s=100, alpha=0.7, c=range(len(selection_df)), cmap='viridis')

for i, row in selection_df.iterrows():
    plt.annotate(row['Method'], 
                (row['Features Selected'], row['R²']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, alpha=0.8)

plt.xlabel('Number of Features Selected')
plt.ylabel('R² Score')
plt.title('Feature Selection: Performance vs Complexity')
plt.grid(True, alpha=0.3)

# Bar plot of R² scores
plt.subplot(1, 2, 2)
colors = ['red' if method == 'Baseline (All)' else 'blue' 
          for method in selection_df['Method']]
bars = plt.bar(range(len(selection_df)), selection_df['R²'], color=colors, alpha=0.7)
plt.xticks(range(len(selection_df)), selection_df['Method'], rotation=45, ha='right')
plt.ylabel('R² Score')
plt.title('Model Performance by Selection Method')

# Add value labels on bars
for bar, r2 in zip(bars, selection_df['R²']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{r2:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## Summary and Best Practices

### Key Takeaways:

1. **Feature Scaling**: Essential for distance-based algorithms (KNN, SVM)
2. **Polynomial Features**: Can capture non-linear relationships but risk overfitting
3. **Categorical Encoding**: Choice depends on cardinality and relationship with target
4. **Feature Selection**: Reduces overfitting and improves interpretability
5. **DateTime Features**: Extract cyclical patterns and temporal relationships

### Best Practices:

- Always split data before feature engineering to prevent data leakage
- Use cyclical encoding for temporal features
- Consider domain knowledge when creating features
- Monitor overfitting when adding complex features
- Use feature selection to reduce dimensionality
- Validate feature importance and interpretability

## Predicting the Survival Rate of Passengers on the Titanic Ship

Download the dataset from: https://github.com/ankitsahu84/titanic

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Jun 30 15:02:50 2020

@author: Ankit Sahu
https://www.kaggle.com/bhatiashivam/ahoy-top-3-the-only-notebook-you-need

"""
#Import the required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [ ]:
#read data from test.csv file into a pandas dataframe
t_train = pd.read_csv("train.csv")
t_test = pd.read_csv("test.csv")

#create a submission array which will hold the passenger ids of the values we need to hold
submission = pd.DataFrame({"PassengerId": t_test["PassengerId"],'Survived': 0 })

#Lets delete the the non relevant columns
del t_train['PassengerId'], t_train['Ticket']
del t_test['PassengerId'], t_test['Ticket']

In [26]:
# Task 1: Basic Data Exploration

# Display dataset dimensions and structure
print("Dataset shapes:")
print(f"Training set: {t_train.shape}")
print(f"Test set: {t_test.shape}")

# Examine first few records
print("\nFirst 5 training records:")
print(t_train.head())

# Check data types
print("\nData types:")
print(t_train.dtypes)

# Check for duplicates
print(f"\nDuplicates in training: {t_train.duplicated().sum()}")
print(f"Duplicates in test: {t_test.duplicated().sum()}")

# Missing values analysis
print("\nMissing values in training:")
print(t_train.isnull().sum())
print("\nMissing values in test:")
print(t_test.isnull().sum())


Dataset shapes:


NameError: name 't_train' is not defined

In [ ]:
## Task 2: Survival Analysis Visualizations

# Basic survival distribution
plt.figure(figsize=(12, 8))

plt.subplot(2, 3, 1)
t_train.Survived.value_counts(normalize=True).plot(kind="bar")
plt.title("Survival Rate Distribution")

plt.subplot(2, 3, 2)
t_train.Pclass.value_counts(normalize=True).plot(kind="bar")
plt.title("Passenger Class Distribution")

plt.subplot(2, 3, 3)
plt.scatter(t_train.Survived, t_train.Age, alpha=0.3)
plt.title("Age vs Survival")

plt.tight_layout()
plt.show()

In [ ]:
## Task 3: Feature-wise Distribution Analysis

# Age distribution by passenger class
plt.figure(figsize=(10, 6))
for pclass in [1, 2, 3]:
  t_train.Age[t_train.Pclass == pclass].plot(kind="kde", label=f"Class {pclass}")
plt.title("Age Distribution by Passenger Class")
plt.legend()
plt.show()

# Embarked location analysis
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
t_train.Embarked.value_counts(normalize=True).plot(kind="bar")
plt.title("Embarked Distribution")

plt.subplot(1, 2, 2)
for port in ['S', 'C', 'Q']:
  t_train.Age[t_train.Embarked == port].plot(kind="kde", label=port)
plt.title("Age Distribution by Embarked Port")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Task 4: Advanced Multi-variable Analysis

# Gender and survival analysis
g = sns.FacetGrid(t_train, col="Sex", row="Survived", margin_titles=True)
g.map(plt.hist, "Age", color="green")
plt.show()

# Class and survival analysis
g = sns.FacetGrid(t_train, col="Sex", row="Survived", margin_titles=True)
g.map(plt.hist, "Pclass", color="purple")
plt.show()

# Fare, age, and survival relationship
g = sns.FacetGrid(t_train, hue="Survived", col="Pclass", margin_titles=True,
                palette={1:"seagreen", 0:"red"})
g.map(plt.scatter, "Fare", "Age", edgecolor="w").add_legend()
plt.show()



In [ ]:
# Task 5: Survival Rate Analysis by Categories

# Survival rates by different factors
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# By embarked location
sns.barplot(x='Embarked', y='Survived', data=t_train, ax=axes[0,0])
axes[0,0].set_title('Survival Rate by Embarked Location')

# By passenger class
sns.barplot(x='Pclass', y='Survived', data=t_train, ax=axes[0,1])
axes[0,1].set_title('Survival Rate by Passenger Class')

# By gender
sns.barplot(x='Sex', y='Survived', data=t_train, ax=axes[0,2])
axes[0,2].set_title('Survival Rate by Gender')

# By siblings/spouses
sns.barplot(x='SibSp', y='Survived', data=t_train, ax=axes[1,0])
axes[1,0].set_title('Survival Rate by Siblings/Spouses')

# By parents/children
sns.barplot(x='Parch', y='Survived', data=t_train, ax=axes[1,1])
axes[1,1].set_title('Survival Rate by Parents/Children')

plt.tight_layout()
plt.show()

In [ ]:
# Task 6: Feature Engineering - Title Extraction

import re

def extract_title(name):
  """Extract title from passenger name"""
  return re.search(", (.*?)\.", name).group(1)

# Apply title extraction
t_train['Title'] = t_train['Name'].apply(extract_title)
t_test['Title'] = t_test['Name'].apply(extract_title)

# Check title distribution
print("Title distribution:")
print(t_train['Title'].value_counts())

# Remove unnecessary columns
t_train.drop(['Name', 'Cabin'], axis=1, inplace=True)
t_test.drop(['Name', 'Cabin'], axis=1, inplace=True)

In [27]:
# Task 7: Data Cleaning and Imputation

# Fill missing values
t_train['Embarked'].fillna(t_train['Embarked'].mode()[0], inplace=True)
t_test['Fare'].fillna(t_test['Fare'].median(), inplace=True)

# Handle rare titles
from feature_engine.encoding import RareLabelEncoder

rare_encoder = RareLabelEncoder(tol=0.05, n_categories=3)
t_train['Title'] = rare_encoder.fit_transform(t_train[['Title']])['Title']
t_test['Title'] = rare_encoder.transform(t_test[['Title']])['Title']

# Create family size feature
t_train['FamilySize'] = t_train['SibSp'] + t_train['Parch'] + 1
t_test['FamilySize'] = t_test['SibSp'] + t_test['Parch'] + 1

# Create is_alone feature
t_train['IsAlone'] = (t_train['FamilySize'] == 1).astype(int)
t_test['IsAlone'] = (t_test['FamilySize'] == 1).astype(int)


NameError: name 't_train' is not defined

In [ ]:
# Task 8: Categorical Encoding

# Encode categorical variables
# Gender encoding
t_train['Sex'] = t_train['Sex'].map({'male': 0, 'female': 1})
t_test['Sex'] = t_test['Sex'].map({'male': 0, 'female': 1})

# Embarked encoding
embarked_map = {'S': 0, 'C': 1, 'Q': 2}
t_train['Embarked'] = t_train['Embarked'].map(embarked_map)
t_test['Embarked'] = t_test['Embarked'].map(embarked_map)

# Title encoding
title_map = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
t_train['Title'] = t_train['Title'].map(title_map)
t_test['Title'] = t_test['Title'].map(title_map)


In [ ]:
# Task 9: Advanced Missing Value Imputation

# Age imputation by title
titles = ['Master', 'Miss', 'Mr', 'Mrs', 'Rare']

for i, title in enumerate(titles):
  # Get median age for each title
  age_median = t_train.groupby('Title')['Age'].median()[i]

  # Fill missing ages
  t_train.loc[(t_train['Age'].isnull()) & (t_train['Title'] == i), 'Age'] = age_median
  t_test.loc[(t_test['Age'].isnull()) & (t_test['Title'] == i), 'Age'] = age_median

# KNN imputation for remaining missing values
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3)
t_train_imputed = pd.DataFrame(imputer.fit_transform(t_train),
                            columns=t_train.columns,
                            index=t_train.index)
t_test_imputed = pd.DataFrame(imputer.transform(t_test),
                           columns=t_test.columns,
                           index=t_test.index)



In [ ]:
# Task 10: Feature Selection and Analysis

# Create fare bands
t_train['FareBand'] = pd.qcut(t_train['Fare'], 4, labels=[1, 2, 3, 4])
t_test['FareBand'] = pd.qcut(t_test['Fare'], 4, labels=[1, 2, 3, 4])

# Feature selection using chi-square test
from sklearn.feature_selection import chi2

y = t_train['Survived']
X = t_train.drop('Survived', axis=1)

# Chi-square analysis
chi_scores, p_values = chi2(X, y)
feature_scores = pd.DataFrame({
  'Feature': X.columns,
  'Chi2_Score': chi_scores,
  'P_Value': p_values
}).sort_values('Chi2_Score', ascending=False)

print("Feature importance by Chi-square test:")
print(feature_scores)



In [ ]:
# Task 11: Model Training and Evaluation

# Prepare final dataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Remove low-importance features based on analysis
features_to_drop = ['Parch', 'SibSp', 'FamilySize', 'Fare', 'Age', 'FareBand']
X_final = X.drop(features_to_drop, axis=1)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# Split data
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

models = {
  'Logistic Regression': LogisticRegression(),
  'Random Forest': RandomForestClassifier(),
  'SVM': SVC(probability=True),
  'XGBoost': xgb.XGBClassifier()
}



In [ ]:
# Task 12: Model Comparison and Final Prediction

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

# Evaluate models
results = []

for name, model in models.items():
  # Train model
  model.fit(X_train, y_train)

  # Predictions
  y_pred = model.predict(X_val)

  # Cross-validation score
  cv_scores = cross_val_score(model, X_scaled, y, cv=5)

  results.append({
      'Model': name,
      'Accuracy': accuracy_score(y_val, y_pred),
      'CV_Mean': cv_scores.mean(),
      'CV_Std': cv_scores.std()
  })

# Display results
results_df = pd.DataFrame(results).sort_values('CV_Mean', ascending=False)
print("Model Performance Comparison:")
print(results_df)

# Train best model on full dataset
best_model = xgb.XGBClassifier()
best_model.fit(X_scaled, y)

# Make final predictions
test_scaled = scaler.transform(t_test_imputed.drop(features_to_drop, axis=1))
predictions = best_model.predict(test_scaled)

# Create submission file
submission = pd.DataFrame({
  'PassengerId': test_ids,  # Assuming you have test passenger IDs
  'Survived': predictions
})
submission.to_csv('titanic_submission.csv', index=False)

